In [4]:
import numpy as np
import torch
import os

from pydosert.data import Patient, MachineConfig, BeamSequence
from pydosert import DoseEngine

In [6]:
machine_config = MachineConfig(
    preset="/home/sglatz54/projects/autoplan_2/PyDoseRT/src/pydose_rt/data/machine_presets/varian_10MV.json",
    profile_corrections=None,  # TODO: add profile corrections
    output_factors=None,
    head_scatter_amplitude=None,
    head_scatter_sigma=None,
    mlc_transmission=0.0
    )
number_of_beams = 180
gantry_angles = torch.from_numpy(np.linspace(-180, 180, number_of_beams, endpoint=False))
field_size = (400, 400)
iso_center = (200.0, 200.0, 200.0)
collimator_angles = torch.from_numpy(np.array([0.0 for _ in range(number_of_beams)]))
sid = 1000.0
open_field_size = 0.0
kernel_size = 15
device = torch.device("cuda")
dtype = torch.float16
resolution = (1.25, 3.125, 3.125)
image_shape = (320, 128, 128)

beam_sequence = BeamSequence.create(gantry_angles,
                                    machine_config.number_of_leaf_pairs,
                                    field_size,
                                    iso_center,
                                    collimator_angles,
                                    sid,
                                    open_field_size,
                                    device,
                                    dtype,
                                    True    
)

engine = DoseEngine(
    machine_config=machine_config,
    dose_grid_spacing=resolution,
    dose_grid_shape=image_shape,
    beam_template=beam_sequence,
    kernel_size=kernel_size, 
    dtype=dtype, 
    device=device
)

engine.calibrate(
    calibration_mu=machine_config.calibration_mu,
    original_beam_template=beam_sequence
)

The argument `original_beam_template` is now deprecated and will not be used for calibration
Calibration failed. Adjusting calibration factor to: 0.0024691358024691358


In [ ]:
from pydosert.physics.attenuation.hu_density_conversion import convert_HU_to_density


path = "/home/sglatz54/projects/rt_ai_preprocesssing/tests/example_data/processed_small/"

for pat in os.listdir(path)[0:1]:
    # hotfix
    path = "/home/sglatz54/projects/rt_ai_preprocesssing/tests/example_data/processed_old/"
    pat = "MrAlderson_test_loc_large_1"
    print(f"Processing patient: {pat}")
    ct = np.load(os.path.join(path, pat, "CT.npy"), allow_pickle=True)
    dose = np.load(os.path.join(path, pat, "Dose.npy"), allow_pickle=True)
    plan = np.load(os.path.join(path, pat, "Beam.npy"), allow_pickle=True).item()

    ct = torch.from_numpy(ct).to(device).unsqueeze(0).to(torch.float16)
    ct = convert_HU_to_density(ct)

    leaves = torch.from_numpy(plan["POS"]).to(device).unsqueeze(0)
    jaws = torch.from_numpy(plan["ASYM"]).to(device).unsqueeze(0)
    metersets = torch.from_numpy(plan["METERSET"]).to(device).unsqueeze(0)

    leaves = leaves.squeeze(-1)
    leaves = leaves.permute(0, 2, 3, 1)
    leaves = leaves.flip(1)

    mus = metersets.squeeze(dim=(1, 3, 4))
    mus = mus.flip(1)

    jaws = jaws.squeeze(dim=(3, 4))
    jaws = jaws.permute(0, 2, 1)
    jaws = jaws.flip(1)

    leaves, jaws, mus = (
        leaves.to(torch.float16),
        jaws.to(torch.float16),
        mus.to(torch.float16),
    )  # Dose engine needs float16

    dose_pred = engine.forward(
        leaves,
        mus,
        jaws,
        ct,
    ).to(torch.float32)

FileNotFoundError: [Errno 2] No such file or directory: '/home/sglatz54/projects/rt_ai_preprocesssing/tests/example_data/processed_small/'

: 

In [ ]:
import torch
from pydosert.data import Phantom
from pydosert.utils.plotting import plot_overview

ph = Phantom.from_uniform_water(shape=image_shape, spacing=resolution)
ph.dose = torch.from_numpy(dose)
ph.add_mask("External", torch.ones(image_shape, dtype=torch.bool), overwrite=True)
plot_overview(ph, dose_pred=dose_pred[0].cpu())

-----------------------------------------------------------------------------------------------------

In [ ]:
machine_config = MachineConfig(
    preset="/home/sglatz54/projects/autoplan_2/PyDoseRT/src/pydose_rt/data/machine_presets/vienna_10MV.json",
    profile_corrections=None,  # TODO: add profile corrections
    output_factors=None,
    head_scatter_amplitude=None,
    head_scatter_sigma=None,
    mlc_transmission=0.0
    )
number_of_beams = 180
gantry_angles = torch.from_numpy(np.linspace(-180, 180, number_of_beams, endpoint=False))
field_size = (400, 400)
iso_center = (200.0, 200.0, 200.0)
collimator_angles = torch.from_numpy(np.array([0.0 for _ in range(number_of_beams)]))
sid = 1000.0
open_field_size = 0.0
kernel_size = 15
device = torch.device("cuda")
dtype = torch.float16
resolution = (3.125, 3.125, 3.125)
image_shape = (128, 128, 128)

beam_sequence = BeamSequence.create(gantry_angles,
                                    machine_config.number_of_leaf_pairs,
                                    field_size,
                                    iso_center,
                                    collimator_angles,
                                    sid,
                                    open_field_size,
                                    device,
                                    dtype,
                                    True    
)

engine = DoseEngine(
    machine_config=machine_config,
    dose_grid_spacing=resolution,
    dose_grid_shape=image_shape,
    beam_template=beam_sequence,
    kernel_size=kernel_size, 
    adjust_values=False,
    dtype=dtype, 
    device=device
)

engine.calibrate(
    calibration_mu=machine_config.calibration_mu,
    original_beam_template=beam_sequence
)

In [ ]:
from pydose_rt.physics.attenuation.hu_density_conversion import convert_HU_to_density


path = "/home/sglatz54/projects/rt_ai_preprocesssing/tests/example_data/processed_small/"

for pat in os.listdir(path)[0:1]:
    # hotfix
    path = "/home/sglatz54/projects/rt_ai_preprocesssing/tests/example_data/processed_old/"
    pat = "MrAlderson_test_loc_small_1"
    print(f"Processing patient: {pat}")
    ct = np.load(os.path.join(path, pat, "CT.npy"), allow_pickle=True)
    dose = np.load(os.path.join(path, pat, "Dose.npy"), allow_pickle=True)
    plan = np.load(os.path.join(path, pat, "Beam.npy"), allow_pickle=True).item()

    ct = torch.from_numpy(ct).to(device).unsqueeze(0).to(torch.float16)
    ct = convert_HU_to_density(ct)

    leaves = torch.from_numpy(plan["POS"]).to(device).unsqueeze(0)
    jaws = torch.from_numpy(plan["ASYM"]).to(device).unsqueeze(0)
    metersets = torch.from_numpy(plan["METERSET"]).to(device).unsqueeze(0)

    leaves = leaves.squeeze(-1)
    leaves = leaves.permute(0, 2, 3, 1)
    leaves = leaves.flip(1)

    mus = metersets.squeeze(dim=(1, 3, 4))
    mus = mus.flip(1)

    jaws = jaws.squeeze(dim=(3, 4))
    jaws = jaws.permute(0, 2, 1)
    jaws = jaws.flip(1)

    leaves, jaws, mus = (
        leaves.to(torch.float16),
        jaws.to(torch.float16),
        mus.to(torch.float16),
    )  # Dose engine needs float16

    dose_pred = engine.forward(
        leaves,
        mus,
        jaws,
        ct,
    ).to(torch.float32)

In [ ]:
import torch
from pydosert.data import Phantom

ph = Phantom.from_uniform_water(shape=image_shape, spacing=resolution)
ph.dose = torch.from_numpy(dose)
ph.add_mask("External", torch.ones(image_shape, dtype=torch.bool), overwrite=True)
plot_overview(ph, dose_pred=dose_pred[0].cpu())